# Fine-tune PP-OCRv5 mobile rec trên crop biển số Việt Nam

**Mục tiêu:** nâng độ chính xác chuỗi biển 2 dòng (hiện ~0,60 trên tập đánh giá 2.801 mẫu)
bằng fine-tune model recognition trên đúng phân phối production (strip 2-dòng-ghép-ngang, cao 64 px).

**Chuẩn bị:** `rec_finetune.zip` (~40 MB, 6.672 train + 571 val) đã nằm ở `MyDrive/DATN/`.

**Runtime:** Thời gian chạy → Thay đổi loại thời gian chạy → **T4 GPU**.

Chạy tuần tự từng cell. Mốc kiểm tra ghi ở đầu mỗi cell.


In [ ]:
# 1) Kiem tra GPU — PHAI in ra 'GPU 0: Tesla T4 ...'
#    Neu khong: doi runtime sang T4 GPU roi chay lai tu dau
!nvidia-smi -L


In [ ]:
# 2a) Cai PaddlePaddle GPU 3.0 — chi co tren index cua Paddle (PyPI dung o 2.6.2)
#     CDN nay hay timeout: --retries 8 de pip tu thu lai, cu de no chay vai phut
#     MOC: dong cuoi phai in 'CUDA: True'
!pip install -q --timeout 300 --retries 8 paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
import paddle; print('CUDA:', paddle.device.is_compiled_with_cuda())


In [ ]:
# 2b) DU PHONG — chi chay neu 2a timeout mai khong xong.
#     wget -c noi tiep phan da tai do: dut mang thi CHAY LAI CHINH CELL NAY
import re, urllib.request
html = urllib.request.urlopen('https://www.paddlepaddle.org.cn/packages/stable/cu118/paddlepaddle-gpu/').read().decode()
name = re.search(r'paddlepaddle_gpu-3\.0\.0[^"]*cp312[^"]*linux_x86_64\.whl', html).group(0)
url = f'https://paddle-whl.bj.bcebos.com/stable/cu118/paddlepaddle-gpu/{name}'
print(url)
!wget -c -q --show-progress --tries=10 --timeout=60 {url}
!pip install -q paddlepaddle_gpu-3.0.0*.whl
import paddle; print('CUDA:', paddle.device.is_compiled_with_cuda())


In [ ]:
# 2c) Clone PaddleOCR (ma nguon training) + requirements
%cd /content
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd /content/PaddleOCR
!pip install -q --timeout 180 --retries 5 -r requirements.txt


In [ ]:
# 3) Mount Drive va giai nen dataset
#    MOC: 6672 train.txt / 571 val.txt
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/data && mkdir -p /content/data
!unzip -q /content/drive/MyDrive/DATN/rec_finetune.zip -d /content/data
!wc -l /content/data/rec_finetune/train.txt /content/data/rec_finetune/val.txt
!head -2 /content/data/rec_finetune/train.txt


In [ ]:
# 4) Tai pretrained weights cua en_PP-OCRv5_mobile_rec
#    MOC: file .pdparams dung luong vai chuc MB (vai KB nghia la trang loi)
#    Neu 404: tra 'PP-OCRv5 model list' trong docs PaddleOCR lay link moi
!mkdir -p /content/pretrained
!wget -c -q --show-progress --tries=10 --timeout=60 -O /content/pretrained/en_PP-OCRv5_mobile_rec_pretrained.pdparams \
    https://paddleocr.bj.bcebos.com/PP-OCRv5/english/en_PP-OCRv5_mobile_rec_pretrained.pdparams
!ls -la /content/pretrained


In [ ]:
# 5) Tim file config cua model trong repo PaddleOCR
#    MOC: in ra it nhat 1 duong dan .yml; neu rong -> bao lai kem ket qua ls
import glob
candidates = glob.glob('configs/rec/**/*en_PP-OCRv5_mobile*.yml', recursive=True)
print(candidates)
if not candidates:
    get_ipython().system('ls configs/rec/')
CONFIG = candidates[0]
print('CONFIG =', CONFIG)


In [ ]:
# 6) Huan luyen (fine-tune) — khoang 1-2 gio tren T4
#    - dict36: charset 36 ky tu (quyet dinh Phase 1, cam thu hep)
#    - max_text_length=10: bien VN dai nhat 9 ky tu
#    MOC: log 'acc' tren eval tang dan; giu tab mo keo Colab ngat phien
!python tools/train.py -c {CONFIG} \
  -o Global.pretrained_model=/content/pretrained/en_PP-OCRv5_mobile_rec_pretrained \
     Global.character_dict_path=/content/data/rec_finetune/dict36.txt \
     Global.use_space_char=False \
     Global.max_text_length=10 \
     Global.epoch_num=30 \
     Global.save_epoch_step=5 \
     Global.eval_batch_step='[0,200]' \
     Global.save_model_dir=/content/output/rec_vn \
     Optimizer.lr.learning_rate=0.0001 \
     Train.dataset.data_dir=/content/data/rec_finetune \
     Train.dataset.label_file_list='[/content/data/rec_finetune/train.txt]' \
     Train.loader.batch_size_per_card=128 \
     Eval.dataset.data_dir=/content/data/rec_finetune \
     Eval.dataset.label_file_list='[/content/data/rec_finetune/val.txt]'


In [ ]:
# 6b) Sao luu checkpoint len Drive NGAY sau khi train xong
#     (Colab free hay ngat phien — mat /content la mat trang cong train)
!mkdir -p /content/drive/MyDrive/DATN/rec_vn_ckpt
!cp /content/output/rec_vn/best_accuracy.* /content/drive/MyDrive/DATN/rec_vn_ckpt/ || true
!cp /content/output/rec_vn/latest.* /content/drive/MyDrive/DATN/rec_vn_ckpt/ || true
!ls -la /content/drive/MyDrive/DATN/rec_vn_ckpt


In [ ]:
# 7) Danh gia checkpoint tot nhat tren tap val sach
#    MOC: GHI LAI con so acc — do la so mang ve local de doi chieu
!python tools/eval.py -c {CONFIG} \
  -o Global.checkpoints=/content/output/rec_vn/best_accuracy \
     Global.character_dict_path=/content/data/rec_finetune/dict36.txt \
     Global.use_space_char=False \
     Global.max_text_length=10 \
     Eval.dataset.data_dir=/content/data/rec_finetune \
     Eval.dataset.label_file_list='[/content/data/rec_finetune/val.txt]'


In [ ]:
# 8) Export inference model va luu ve Drive
#    MOC: rec_vn_inference.zip xuat hien trong MyDrive/DATN/
!python tools/export_model.py -c {CONFIG} \
  -o Global.checkpoints=/content/output/rec_vn/best_accuracy \
     Global.character_dict_path=/content/data/rec_finetune/dict36.txt \
     Global.use_space_char=False \
     Global.max_text_length=10 \
     Global.save_inference_dir=/content/output/rec_vn_inference
!cd /content/output && zip -r -q rec_vn_inference.zip rec_vn_inference
!cp /content/output/rec_vn_inference.zip /content/drive/MyDrive/DATN/
!ls -la /content/drive/MyDrive/DATN/rec_vn_inference.zip


## Tích hợp về hệ thống (làm ở máy local)

1. Tải `rec_vn_inference.zip` từ Drive, giải nén vào `D:\DATN\models\rec_finetuned\`
   (các file inference nằm NGAY trong thư mục đó, không lồng thêm cấp).
2. Bật bằng một biến môi trường (mọi dây nối đã sẵn):
```
# Docker (.env):  ALPR_OCR_REC_MODEL_DIR=/app/models/rec_finetuned
# Local:          set ALPR_OCR_REC_MODEL_DIR=models/rec_finetuned
```
3. **Nghi thức đo bắt buộc** trước khi công bố số (ai/training/README-rec-finetune.md):
   A4–A7 toàn tập so baseline · 3 bộ hồi quy · ablation bật/tắt.
   Chỉ tiêu: **A6 biển 2 dòng tăng ≥ 5 điểm, biển 1 dòng không giảm** — không đạt thì gỡ cờ, ghi trung thực.
